# Coherence, Cross-Spectra, and Spike–LFP Coupling — Reference Notebook

**Neural Data Science (WP7)** · LMU Biology

---

This notebook is a self-contained reference for **bivariate spectral
analysis** and **spike–LFP coupling** applied to hippocampal recordings.
It covers:

1. **Cross-spectral density (CSD)** — measuring frequency-resolved
   coupling between two LFP channels
2. **Coherency and coherence** — normalised CSD magnitude and phase
3. **Laminar coherence profiles** — theta/gamma structure across the shank
4. **Spike–LFP phase locking** — circular statistics and the Rayleigh test
5. **Spike-triggered averages** and **phase histograms**
6. **Spike–LFP coherence** — frequency-resolved spike–field coupling

All code uses `numpy`, `scipy`, and `matplotlib` only (plus `wp7_helpers`
for multitaper utilities).

In [ ]:
import sys
import os

import numpy as np
import matplotlib.pyplot as plt
from scipy import io, signal
from scipy.signal import butter, filtfilt, hilbert, coherence, csd, welch

sys.path.insert(0, '../lib')
from wp7_helpers import cross_spectrum

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (10, 4)

In [ ]:
# Load LFP and spike data
ds = '/storage2/arash/sirocampus/data/ds-wp7'

lfp_path = os.path.join(ds, 'ws_data_1shank.mat')
spk_path = os.path.join(ds, 'spikes.mat')
for p in [lfp_path, spk_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(f'Data not found: {p}')

mat = io.loadmat(lfp_path)
lfps = mat['lfps'].astype(np.float64)
fs = 1250.0
n_samples, n_channels = lfps.shape

mat_spk = io.loadmat(spk_path)
res = mat_spk['Res'].ravel()   # spike times in samples (1-indexed)
clu = mat_spk['Clu'].ravel()   # cluster ID per spike
clusters = np.unique(clu)

print(f'LFP: {lfps.shape}  ({n_samples / fs:.1f} s, {n_channels} ch)')
print(f'Spikes: {len(res)}  |  Clusters: {clusters}')

## 1. Cross-spectral density and coherency

### Theory

The **coherency** between two signals $x$ and $y$ is the normalised
cross-spectral density:

$$
C_{xy}(f) = \frac{S_{xy}(f)}{\sqrt{S_{xx}(f)\,S_{yy}(f)}}
$$

where $S_{xy}$ is the cross-spectral density and $S_{xx}$, $S_{yy}$ are
auto-spectra.  Coherency is **complex-valued**:

- **Magnitude** $|C_{xy}|$ — strength of linear coupling (0 = independent,
  1 = perfectly correlated at that frequency)
- **Phase** $\arg C_{xy}$ — consistent phase relationship between channels

The **magnitude-squared coherence** $|C_{xy}|^2$ is the quantity most
commonly reported.

In [ ]:
# Compute full CSD matrix using Welch-windowed CSD (2 s windows, 75% overlap)
nperseg = int(2.0 * fs)  # 2 s windows
noverlap = int(0.75 * nperseg)

f_csd = None
csd_full = None

for i in range(n_channels):
    for j in range(i, n_channels):
        f, Pxy = csd(lfps[:, i], lfps[:, j], fs=fs, nperseg=nperseg,
                      noverlap=noverlap, return_onesided=True)
        if csd_full is None:
            f_csd = f
            csd_full = np.zeros((n_channels, n_channels, len(f)), dtype=complex)
        csd_full[i, j, :] = Pxy
        csd_full[j, i, :] = np.conj(Pxy)  # Hermitian symmetry

# Auto-spectra and coherency
auto_spectra = np.real(np.array([csd_full[ch, ch, :] for ch in range(n_channels)]))

coherency = np.zeros_like(csd_full)
for i in range(n_channels):
    for j in range(n_channels):
        coherency[i, j, :] = csd_full[i, j, :] / np.sqrt(
            auto_spectra[i] * auto_spectra[j] + 1e-30
        )

coh_mag = np.abs(coherency)
coh_phase = np.angle(coherency)

print(f'CSD matrix: {csd_full.shape}  ({len(f_csd)} freq bins, '
      f'Δf = {f_csd[1]:.2f} Hz)')

### Laminar structure: theta and gamma coherence

For a hippocampal linear probe we expect:

- **Theta (6–10 Hz):** high coherence across the full shank with a
  characteristic ~180° phase reversal near the pyramidal cell layer
  (current sink → source transition).
- **Gamma (30–80 Hz):** coherence confined to nearby channels within
  specific laminar compartments.

In [ ]:
def band_mean(data, freqs, flo, fhi):
    """Average data[..., n_freqs] across a frequency band."""
    mask = (freqs >= flo) & (freqs <= fhi)
    return np.mean(data[..., mask], axis=-1)

theta_mag = band_mean(coh_mag, f_csd, 6, 10)
theta_phase = band_mean(coh_phase, f_csd, 6, 10)
gamma_mag = band_mean(coh_mag, f_csd, 30, 80)
gamma_phase = band_mean(coh_phase, f_csd, 30, 80)

fig, axes = plt.subplots(2, 2, figsize=(10, 9))
ch_ax = np.arange(n_channels)

im0 = axes[0, 0].imshow(theta_mag, cmap='hot', aspect='equal',
                          extent=[0, n_channels-1, n_channels-1, 0])
axes[0, 0].set_title('Theta coherence magnitude')
plt.colorbar(im0, ax=axes[0, 0])

im1 = axes[0, 1].imshow(np.degrees(theta_phase), cmap='twilight',
                          aspect='equal', vmin=-180, vmax=180,
                          extent=[0, n_channels-1, n_channels-1, 0])
axes[0, 1].set_title('Theta coherence phase (deg)')
plt.colorbar(im1, ax=axes[0, 1])

im2 = axes[1, 0].imshow(gamma_mag, cmap='hot', aspect='equal',
                          extent=[0, n_channels-1, n_channels-1, 0])
axes[1, 0].set_title('Gamma coherence magnitude')
plt.colorbar(im2, ax=axes[1, 0])

im3 = axes[1, 1].imshow(np.degrees(gamma_phase), cmap='twilight',
                          aspect='equal', vmin=-180, vmax=180,
                          extent=[0, n_channels-1, n_channels-1, 0])
axes[1, 1].set_title('Gamma coherence phase (deg)')
plt.colorbar(im3, ax=axes[1, 1])

for ax in axes.flat:
    ax.set(xlabel='Channel j', ylabel='Channel i')

fig.suptitle('Cross-channel coherence matrices', y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# Coherence and phase profile vs a reference channel
ref_ch = 0
freq_mask = (f_csd >= 1) & (f_csd <= 100)

coh_profile = coh_mag[ref_ch, :, :][:, freq_mask]        # (n_channels, n_freqs)
phase_profile = np.degrees(coh_phase[ref_ch, :, :][:, freq_mask])

fig, axes = plt.subplots(2, 1, figsize=(10, 7))

im0 = axes[0].pcolormesh(f_csd[freq_mask], ch_ax, coh_profile,
                           shading='auto', cmap='hot')
axes[0].set(ylabel='Channel', title=f'Coherence magnitude — ref Ch {ref_ch}')
plt.colorbar(im0, ax=axes[0], label='Coherence')

im1 = axes[1].pcolormesh(f_csd[freq_mask], ch_ax, phase_profile,
                           shading='auto', cmap='twilight', vmin=-180, vmax=180)
axes[1].set(xlabel='Frequency (Hz)', ylabel='Channel',
            title=f'Coherence phase — ref Ch {ref_ch}')
plt.colorbar(im1, ax=axes[1], label='Phase (deg)')

fig.tight_layout()
plt.show()

## 2. Spike–LFP phase locking

### Theta phase at spike times

To measure spike–LFP coupling, we:
1. Bandpass-filter the LFP in the theta range (6–10 Hz)
2. Compute the **analytic signal** via the Hilbert transform
3. Read the **instantaneous phase** at each spike timestamp

In [ ]:
# Theta bandpass (6–10 Hz, 4th-order Butterworth, zero-phase)
b_th, a_th = butter(4, [6.0, 10.0], btype='bandpass', fs=fs)

theta_ref_ch = 0  # surface reference channel
theta_filt = filtfilt(b_th, a_th, lfps[:, theta_ref_ch])
theta_analytic = hilbert(theta_filt)
theta_phase_all = np.angle(theta_analytic)

# Phase at each spike time (Res is 1-indexed → convert to 0-indexed)
spike_idx = (res - 1).astype(int)
valid = (spike_idx >= 0) & (spike_idx < n_samples)
spike_idx = spike_idx[valid]
clu_valid = clu[valid]

spike_theta_phase = theta_phase_all[spike_idx]
print(f'Valid spikes: {len(spike_idx)} / {len(res)}')

### Circular statistics: resultant length and Rayleigh test

The **mean resultant length** measures phase concentration:

$$
R = \left| \frac{1}{N} \sum_{k=1}^{N} e^{j\phi_k} \right|
$$

$R = 0$ for uniformly distributed phases (no locking); $R = 1$ for
perfect phase concentration.

The **Rayleigh test** evaluates the null hypothesis of uniform phase
distribution.  Test statistic: $Z = N R^2$.  For large $N$ under the
null, $p \approx e^{-Z}$ (Mardia & Jupp, 2000).

In [ ]:
def resultant_length(phases):
    """Mean resultant length R of a set of circular observations."""
    return np.abs(np.mean(np.exp(1j * phases)))


def rayleigh_test(phases):
    """Rayleigh test: returns (R, Z, p-value)."""
    n = len(phases)
    R = resultant_length(phases)
    Z = n * R ** 2
    # Approximation valid for n >= 10
    p = np.exp(-Z) * (
        1 + (2 * Z - Z**2) / (4 * n)
        - (24 * Z - 132 * Z**2 + 76 * Z**3 - 9 * Z**4) / (288 * n**2)
    )
    return R, Z, max(p, 0.0)


# Per-cluster phase locking
print(f'{"Cluster":>8} {"N spikes":>10} {"R":>8} {"Z":>10} {"p-value":>12}')
print('-' * 55)
for c in clusters:
    mask = clu_valid == c
    phases = spike_theta_phase[mask]
    R, Z, p = rayleigh_test(phases)
    print(f'{c:>8d} {len(phases):>10d} {R:>8.4f} {Z:>10.2f} {p:>12.2e}')

In [ ]:
# Sample size effects on R
n_per_cluster = {int(c): int(np.sum(clu_valid == c)) for c in clusters}
target_cluster = max(n_per_cluster, key=n_per_cluster.get)
target_phases = spike_theta_phase[clu_valid == target_cluster]

rng = np.random.default_rng(42)
subsample_sizes = np.unique(
    np.logspace(0.5, np.log10(len(target_phases)), 30).astype(int)
)
n_repeats = 50

R_mean = np.zeros(len(subsample_sizes))
R_std = np.zeros(len(subsample_sizes))

for i, n_sub in enumerate(subsample_sizes):
    Rs = []
    for _ in range(n_repeats):
        idx = rng.choice(len(target_phases),
                         size=min(n_sub, len(target_phases)), replace=False)
        R, _, _ = rayleigh_test(target_phases[idx])
        Rs.append(R)
    R_mean[i] = np.mean(Rs)
    R_std[i] = np.std(Rs)

fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(subsample_sizes, R_mean - R_std, R_mean + R_std, alpha=0.25)
ax.plot(subsample_sizes, R_mean, 'o-', markersize=3)
ax.set(xlabel='N spikes (subsampled)', ylabel='Resultant length R',
       title=f'Cluster {target_cluster}: R converges with sample size',
       xscale='log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

At small $N$ (< 20–30 spikes), $R$ is **upward-biased** and highly
variable — it may appear significant by chance.  With $> 100$ spikes the
estimate converges to the true population value.

## 3. Spike-triggered averages and phase histograms

The **spike-triggered average (STA)** reveals the mean LFP waveform
around each spike.  For theta-locked units the STA shows a theta
oscillation centred on the spike.

In [ ]:
sta_half_win = int(0.2 * fs)  # ±200 ms
sta_t = np.arange(-sta_half_win, sta_half_win) / fs * 1000  # ms


def compute_sta(lfp_ch, spike_times, half_win):
    """Spike-triggered average: mean ± SEM."""
    valid = (spike_times > half_win) & (spike_times < len(lfp_ch) - half_win)
    snippets = np.array([
        lfp_ch[t - half_win:t + half_win]
        for t in spike_times[valid]
    ])
    return snippets.mean(axis=0), snippets.std(axis=0) / np.sqrt(valid.sum()), valid.sum()


n_clus = len(clusters)
fig, axes = plt.subplots(1, n_clus, figsize=(4 * n_clus, 3.5), sharey=True)
if n_clus == 1:
    axes = [axes]

for ax, c in zip(axes, clusters):
    mask = clu_valid == c
    sta_mean, sta_sem, n_spk = compute_sta(lfps[:, theta_ref_ch],
                                            spike_idx[mask], sta_half_win)
    ax.fill_between(sta_t, sta_mean - sta_sem, sta_mean + sta_sem, alpha=0.3)
    ax.plot(sta_t, sta_mean, 'k-', linewidth=1.2)
    ax.axvline(0, color='red', linestyle='--', alpha=0.5)
    ax.set(xlabel='Time (ms)', title=f'Cluster {c} (n={n_spk})')

axes[0].set_ylabel('LFP (a.u.)')
fig.suptitle(f'Spike-triggered averages — ref Ch {theta_ref_ch}', y=1.03)
fig.tight_layout()
plt.show()

In [ ]:
# Circular phase histograms
n_bins = 24
bin_edges = np.linspace(-np.pi, np.pi, n_bins + 1)
bin_centres = 0.5 * (bin_edges[:-1] + bin_edges[1:])

fig, axes = plt.subplots(1, n_clus, figsize=(4 * n_clus, 3.5),
                          subplot_kw={'projection': 'polar'})
if n_clus == 1:
    axes = [axes]

for ax, c in zip(axes, clusters):
    phases = spike_theta_phase[clu_valid == c]
    counts, _ = np.histogram(phases, bins=bin_edges)
    counts_norm = counts / counts.sum()
    R = resultant_length(phases)

    bars = ax.bar(bin_centres, counts_norm, width=2 * np.pi / n_bins,
                  alpha=0.7, edgecolor='black', linewidth=0.3)
    ax.set_title(f'Cluster {c}\nR = {R:.3f}', pad=12, fontsize=9)

fig.suptitle('Theta phase distribution at spike times', y=1.05)
fig.tight_layout()
plt.show()

## 4. Spike–LFP coherence

Spike–LFP coherence quantifies **frequency-resolved** coupling between a
point process (spike train) and a continuous signal (LFP).  We construct a
binary spike train at the LFP sampling rate, then compute coherence
between the two signals using standard CSD methods.

In [ ]:
def spike_train_to_binary(spike_times, n_samples):
    """Convert spike timestamps (0-indexed) to a binary time series."""
    train = np.zeros(n_samples)
    valid = spike_times < n_samples
    train[spike_times[valid]] = 1.0
    return train


nperseg_slc = int(2.0 * fs)
noverlap_slc = int(0.75 * nperseg_slc)

fig, axes = plt.subplots(1, n_clus, figsize=(5 * n_clus, 4), sharey=True)
if n_clus == 1:
    axes = [axes]

for ax, c in zip(axes, clusters):
    mask = clu_valid == c
    train = spike_train_to_binary(spike_idx[mask], n_samples)

    f_slc, coh_slc = coherence(train, lfps[:, theta_ref_ch], fs=fs,
                                nperseg=nperseg_slc, noverlap=noverlap_slc)
    freq_mask = (f_slc >= 1) & (f_slc <= 100)

    ax.plot(f_slc[freq_mask], coh_slc[freq_mask], linewidth=1.0)
    ax.axvspan(6, 10, alpha=0.10, color='red')
    ax.set(xlabel='Frequency (Hz)',
           title=f'Cluster {c} — spike–LFP coh (Ch {theta_ref_ch})')
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Coherence')
fig.tight_layout()
plt.show()

In [ ]:
# Cross-layer spike–LFP coherence for the most abundant cluster
layer_channels = [0, 4, 8, 12, 15]

# Pick cluster with most spikes
target_clu = max(n_per_cluster, key=n_per_cluster.get)
train = spike_train_to_binary(spike_idx[clu_valid == target_clu], n_samples)

fig, ax = plt.subplots(figsize=(10, 5))
for ch_i in layer_channels:
    f_slc, coh_slc = coherence(train, lfps[:, ch_i], fs=fs,
                                nperseg=nperseg_slc, noverlap=noverlap_slc)
    freq_mask = (f_slc >= 1) & (f_slc <= 50)
    ax.plot(f_slc[freq_mask], coh_slc[freq_mask], label=f'Ch {ch_i}')

ax.axvspan(6, 10, alpha=0.10, color='red')
ax.set(xlabel='Frequency (Hz)', ylabel='Coherence',
       title=f'Cluster {target_clu} — spike–LFP coherence across depth')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

### Cross-channel coherence

- **Theta coherence** is high across the full shank; the **phase** map
  shows a reversal near the pyramidal layer.
- **Gamma coherence** is spatially confined to nearby channels and
  reflects localised gamma generators at specific depths.

### Spike–LFP coupling

- **Resultant length** $R$ quantifies phase locking strength; use the
  **Rayleigh test** to assess significance.
- Small sample sizes inflate $R$ — always check that $N > 50$–100
  spikes before interpreting phase-locking values.
- **Spike-triggered averages** visualise the average LFP waveform
  around spikes; theta-locked cells show clear oscillations.
- **Spike–LFP coherence** reveals at which *frequency* coupling occurs;
  it depends on which LFP channel is used as reference (laminar
  dependence).

### Practical tips

- Use `scipy.signal.csd` or `wp7_helpers.cross_spectrum` for CSD
  computation; segment length controls frequency resolution.
- Always estimate a **null distribution** (trial shuffling or circular
  shift) to assess coherence significance.
- The `wp7_helpers.coherence` function provides multitaper-averaged
  magnitude-squared coherence; see `lib/wp7_helpers.py`.